# Explore kcEXP00H data — interactive

Browse the raw source data (`xarray_files/*.nc`, variable `deltaF_rg`) **before/independent of**
LFADS. The 75 brain regions are **named and identical across all 5 flies**, so you can compare the
same region across flies directly.

Three interactive views (ipywidgets — run the cell, then use the dropdowns/sliders):

1. **All flies, one region, all conditions** — a panel per fly, condition-averaged traces.
2. **All flies overlaid, one region + one condition** — every fly on one axis (mean ± SEM).
3. **One fly, condition heatmaps** — region × time per condition.

Each view has a **raw ΔF/F ↔ z-scored** toggle. Z-scoring here is **per-region, per-fly**
(subtract that region's mean over all trials × timepoints, divide by its std) — the same kind of
transform used to build the `*_zscored` LFADS dataset. Because it's a per-region affine transform,
it only rescales each trace vertically; it does **not** change the temporal shape.

## Setup & load

In [1]:
from glob import glob
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib import cm
import ipywidgets as W
from IPython.display import display

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 9

XARRAY_DIR = ('/media/server/gklab/KarenCheng/DATA/LFM_current/'
              'kcEXP00H/kcEXP00H_PythonNotebooks/xarray_files/')

COND_ORDER = ['VOF', 'VOx', 'VxF', 'Vxx', 'xOF', 'xOx', 'xxF', 'xxx']
COND_COLOR = {c: cm.tab10(i % 10) for i, c in enumerate(COND_ORDER)}

# --- Load every fly ---
nc_paths = sorted(glob(str(Path(XARRAY_DIR) / '*.nc')))
assert nc_paths, f"No .nc files in {XARRAY_DIR}"

def first_on_run(v):
    # (start, end) of the first contiguous >0 run in v, or None
    on = np.ravel(v) > 0
    if not on.any():
        return None
    start = int(np.argmax(on))
    end = start + int(np.argmin(on[start:])) if not on[start:].all() else len(on)
    return (start, end)

flies = []                       # short ids, e.g. 'H30003-005'
raw = {}                         # fly -> (trial, time, region) float ΔF/F
conds = {}                       # fly -> (trial,) str
is_base = {}                     # fly -> (trial,) bool
ODOR_ON = {}                     # fly -> (start, end) frames of first odor-on run
import re
for p in nc_paths:
    ds = xr.open_dataset(p)
    fid = re.search(r'H\d+-\d+', Path(p).stem).group()
    flies.append(fid)
    raw[fid] = ds['deltaF_rg'].transpose('trial', 'time', 'region').values.astype(float)
    conds[fid] = np.asarray(ds['condition'].values).reshape(-1).astype(str)
    is_base[fid] = np.asarray(ds['is_baseline'].values).reshape(-1).astype(bool)
    region_names = [str(r) for r in np.ravel(ds['region'].values)]
    ODOR_ON[fid] = first_on_run(ds['odor_state_frame'].values)
    ds.close()

n_time = raw[flies[0]].shape[1]
print(f"{len(flies)} flies: {flies}")
print(f"regions: {len(region_names)} (shared) | frames/trial: {n_time}")
print("odor-on (first run) per fly:", {f: ODOR_ON[f] for f in flies})
print("conditions per fly:")
for f in flies:
    u, c = np.unique(conds[f], return_counts=True)
    print(f"  {f}: " + " ".join(f"{a}={b}" for a, b in zip(u, c)))

5 flies: ['H30003-005', 'H34006-007', 'H35005-006', 'H36006-007', 'H37003-004']
regions: 75 (shared) | frames/trial: 1050
odor-on (first run) per fly: {'H30003-005': (83, 92), 'H34006-007': (83, 92), 'H35005-006': (83, 92), 'H36006-007': (83, 92), 'H37003-004': (141, 151)}
conditions per fly:
  H30003-005: VOF=2 VOx=2 VxF=2 Vxx=2 xOF=2 xOx=2 xxF=2 xxx=4
  H34006-007: VOF=2 VOx=2 VxF=2 Vxx=2 xOF=2 xOx=2 xxF=2 xxx=4
  H35005-006: VOF=2 VOx=2 VxF=2 Vxx=2 xOF=2 xOx=2 xxF=2 xxx=4
  H36006-007: VOF=2 VOx=2 VxF=2 Vxx=2 xOF=2 xOx=2 xxF=2 xxx=4
  H37003-004: VOF=2 VOx=2 VxF=2 Vxx=2 xOF=2 xOx=2 xxF=2 xxx=4


## Helpers

`get(fly, region_idx, zscore)` returns that region's `(trials, time)` matrix, optionally
z-scored per-region-per-fly. NaNs (present in the raw data) are handled with `nan`-aware means.

In [2]:
def get(fly, ridx, zscore):
    x = raw[fly][:, :, ridx]                      # (trials, time)
    if zscore:
        mu = np.nanmean(x); sd = np.nanstd(x)
        x = (x - mu) / (sd if sd > 0 else 1.0)
    return x

def shade_odor(ax, fly):
    span = ODOR_ON.get(fly)
    if span is not None:
        ax.axvspan(span[0], span[1], color='gold', alpha=0.15, lw=0)

def trial_mask(fly, selected_conds, exclude_baseline):
    m = np.isin(conds[fly], list(selected_conds))
    if exclude_baseline:
        m &= ~is_base[fly]
    return m

REGION_OPTS = [(f"{i:2d}  {n}", i) for i, n in enumerate(region_names)]
ZMODE = {'raw ΔF/F': False, 'z-scored': True}

## View 1 — all flies · one region · all conditions

Condition-averaged trace per fly (one panel each). Optionally overlay faint single trials. The gold
band marks the odor-on window.

In [3]:
def view_all_flies(region, zmode, sel_conds, show_trials, exclude_baseline, share_y):
    z = ZMODE[zmode]
    present = [c for c in COND_ORDER if c in sel_conds]
    ncol = min(len(flies), 3); nrow = int(np.ceil(len(flies) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 2.8 * nrow),
                             sharex=True, sharey=share_y, squeeze=False)
    axes = axes.flatten()
    for ax, fly in zip(axes, flies):
        x = get(fly, region, z)
        for c in present:
            m = (conds[fly] == c) & (~is_base[fly] if exclude_baseline else True)
            if not m.any():
                continue
            if show_trials:
                for tr in np.where(m)[0]:
                    ax.plot(x[tr], color=COND_COLOR[c], alpha=0.18, lw=0.6)
            ax.plot(np.nanmean(x[m], 0), color=COND_COLOR[c], lw=1.6)
        shade_odor(ax, fly)
        ax.set_title(fly, fontsize=9)
    for ax in axes[len(flies):]:
        ax.axis('off')
    handles = [plt.Line2D([], [], color=COND_COLOR[c], label=c) for c in present]
    fig.legend(handles=handles, loc='center right', fontsize=8,
               bbox_to_anchor=(1.08, 0.5))
    units = 'z-scored' if z else 'ΔF/F'
    fig.suptitle(f"Region {region}: {region_names[region]}   ({units})")
    fig.supxlabel('frame'); plt.tight_layout(); plt.show()

W.interact(
    view_all_flies,
    region=W.Dropdown(options=REGION_OPTS, value=0, description='region'),
    zmode=W.ToggleButtons(options=list(ZMODE), description='scale'),
    sel_conds=W.SelectMultiple(options=COND_ORDER, value=tuple(COND_ORDER),
                               description='conds', rows=8),
    show_trials=W.Checkbox(value=False, description='single trials'),
    exclude_baseline=W.Checkbox(value=True, description='drop baseline'),
    share_y=W.Checkbox(value=True, description='shared y-axis'),
);

interactive(children=(Dropdown(description='region', options=((' 0  AME-R', 0), (' 1  LO-R', 1), (' 2  NO', 2)…

## View 2 — all flies overlaid · one region · one condition

Every fly on a single axis (mean across that condition's trials, shaded ± SEM). Best for asking
"does this region behave the same across flies for this stimulus?"

In [4]:
def view_overlay(region, zmode, cond, exclude_baseline):
    z = ZMODE[zmode]
    fig, ax = plt.subplots(figsize=(8, 4))
    colors = cm.viridis(np.linspace(0, 1, len(flies)))
    for fly, col in zip(flies, colors):
        x = get(fly, region, z)
        m = (conds[fly] == cond) & (~is_base[fly] if exclude_baseline else True)
        if not m.any():
            continue
        mu = np.nanmean(x[m], 0)
        sem = np.nanstd(x[m], 0) / max(1, np.sqrt(m.sum()))
        ax.plot(mu, color=col, lw=1.8, label=f"{fly} (n={int(m.sum())})")
        ax.fill_between(np.arange(len(mu)), mu - sem, mu + sem, color=col, alpha=0.15)
        if ODOR_ON.get(fly) is not None:
            ax.axvline(ODOR_ON[fly][0], color=col, lw=0.7, ls=':')   # this fly's odor onset
    units = 'z-scored' if z else 'ΔF/F'
    ax.set_title(f"{region_names[region]} — condition {cond} ({units})")
    ax.set_xlabel('frame'); ax.set_ylabel(units)
    ax.legend(fontsize=8, loc='best'); plt.tight_layout(); plt.show()

W.interact(
    view_overlay,
    region=W.Dropdown(options=REGION_OPTS, value=0, description='region'),
    zmode=W.ToggleButtons(options=list(ZMODE), description='scale'),
    cond=W.Dropdown(options=COND_ORDER, value='VOF', description='condition'),
    exclude_baseline=W.Checkbox(value=True, description='drop baseline'),
);

interactive(children=(Dropdown(description='region', options=((' 0  AME-R', 0), (' 1  LO-R', 1), (' 2  NO', 2)…

## View 3 — one fly · region × time heatmap per condition

Condition-averaged heatmap (regions on y, time on x) for a chosen fly — the population view. Use the
raw/z toggle to see how z-scoring equalizes regions with different baseline magnitudes.

In [5]:
def view_heatmap(fly, zmode, sel_conds, exclude_baseline):
    z = ZMODE[zmode]
    present = [c for c in COND_ORDER if c in sel_conds]
    # build (region, time) condition-averages
    X = raw[fly]                                  # (trial, time, region)
    if z:
        mu = np.nanmean(X, (0, 1), keepdims=True)
        sd = np.nanstd(X, (0, 1), keepdims=True); sd[sd == 0] = 1
        X = (X - mu) / sd
    ncol = min(len(present), 4); nrow = int(np.ceil(len(present) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 3.0 * nrow),
                             squeeze=False)
    axes = axes.flatten()
    vmax = np.nanpercentile(np.abs(X), 98)
    for ax, c in zip(axes, present):
        m = (conds[fly] == c) & (~is_base[fly] if exclude_baseline else True)
        img = np.nanmean(X[m], 0).T if m.any() else np.full((X.shape[2], X.shape[1]), np.nan)
        im = ax.imshow(img, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        if ODOR_ON.get(fly) is not None:
            ax.axvline(ODOR_ON[fly][0], color='k', lw=0.6, ls=':')
        ax.set_title(f"{c} (n={int(m.sum())})", fontsize=9)
        ax.set_xlabel('frame')
    for ax in axes[len(present):]:
        ax.axis('off')
    axes[0].set_ylabel('region')
    fig.suptitle(f"{fly} — {'z-scored' if z else 'ΔF/F'} (region × time per condition)")
    fig.colorbar(im, ax=axes[:len(present)], fraction=0.02)
    plt.show()

W.interact(
    view_heatmap,
    fly=W.Dropdown(options=flies, value=flies[0], description='fly'),
    zmode=W.ToggleButtons(options=list(ZMODE), description='scale'),
    sel_conds=W.SelectMultiple(options=COND_ORDER, value=tuple(COND_ORDER),
                               description='conds', rows=8),
    exclude_baseline=W.Checkbox(value=True, description='drop baseline'),
);

interactive(children=(Dropdown(description='fly', options=('H30003-005', 'H34006-007', 'H35005-006', 'H36006-0…

---
**Notes**
- Data is the **full within-trial window** (1050 frames). The LFADS pipeline used a 175-frame window
  around odor onset (`PRE=25, POST=150`); here you see everything so you can judge baseline/drift
  outside that window.
- `raw` holds native ΔF/F; the z-scored views recompute z **per region per fly** on the fly, so
  nothing is cached in scaled units.
- Regions are shared across flies, so View 1/2 compare the *same brain region* across animals.